# v0.2.0 — Ensemble Simplification Experiment

**Projeto:** Renewable Energy MLOps — Wind Power Forecasting  
**Status:** experimento em desenvolvimento  
**Branch esperada:** `experiment/ensemble-simplification`

Este notebook documenta a comparação controlada entre:

- **Baseline:** `LGBM + XGBoost + Random Forest`
- **Candidato simplificado:** `LGBM + Random Forest`

A motivação é uma observação anterior de peso aproximadamente zero para o XGBoost no meta-learner. Aqui isso é tratado como **hipótese experimental**, não como justificativa suficiente para removê-lo.

## 1. Objetivo

Avaliar se a remoção do XGBoost reduz a complexidade do ensemble sem degradar materialmente a qualidade preditiva.

As duas arquiteturas devem usar:

- o mesmo snapshot temporal;
- o mesmo contrato de 9 features;
- a mesma separação treino/OOT;
- o mesmo `TimeSeriesSplit` externo;
- o mesmo `TimeSeriesSplit` interno;
- o mesmo orçamento de trials Optuna;
- a mesma métrica primária de otimização: **nMAE (%) em MW**;
- as mesmas métricas OOT.

## 2. Hipótese

> **H0:** o XGBoost ainda acrescenta informação útil ao ensemble sob a janela temporal expandida.

> **H1:** `LGBM + RF` preserva desempenho comparável a `LGBM + XGB + RF`, com menor complexidade operacional.

A decisão deve ser baseada no experimento com a nova janela temporal.

## 3. Contrato experimental

```text
FULL
└── lgbm + xgboost + rf

SIMPLIFIED
└── lgbm + rf
```

O optimizer deve criar um espaço de busca específico para cada arquitetura. Se `xgboost` estiver desabilitado, nenhum hiperparâmetro `xgb_*` deve existir no estudo.

In [ ]:
from energy_mlops.models.ensemble_config import (
    validate_enabled_estimators,
)
from energy_mlops.models.optimize_stacking_ensemble import run_optimization
from energy_mlops.models.train_stacking_ensemble import train_stacking_regressor

ARCHITECTURES = {
    "full": validate_enabled_estimators(
        ("lgbm", "xgboost", "rf")
    ),
    "simplified": validate_enabled_estimators(
        ("lgbm", "rf")
    ),
}

ARCHITECTURES

## 4. Configuração do benchmark

Os valores abaixo devem permanecer idênticos entre as duas arquiteturas.

A flag `RUN_EXPERIMENTS` evita iniciar uma otimização longa acidentalmente.

In [ ]:
EXPERIMENT_CONFIG = {
    "n_trials": 20,
    "n_splits": 3,
    "stacking_n_splits": 5,
}

RUN_EXPERIMENTS = False

EXPERIMENT_CONFIG

## 5. Proveniência de Dados e Correção Temporal — v0.2.0

Antes de avaliar as arquiteturas simplificadas de ensemble, a camada de dados foi revisada para fortalecer a proveniência das fontes e a consistência temporal.

### Geração eólica — ONS

A geração eólica horária para a Bahia é extraída dos dados abertos do ONS.

O extrator agora trata os valores de geração ausentes de forma explícita:

* strings vazias são interpretadas como valores ausentes, não como `0 MW`;
* se algum registro no nível da usina estiver ausente dentro de uma hora, a hora inteira é considerada incompleta;
* horas incompletas são excluídas antes da agregação no nível estadual;
* somas parciais no nível da usina, portanto, nunca são apresentadas como a geração total da Bahia.

Isso evita a introdução de valores artificiais de geração zero ou alvos estaduais incompletos.

### Clima — Open-Meteo

As variáveis meteorológicas continuam usando Morro do Chapéu, Bahia, como um proxy climático regional.

Esta é uma aproximação de modelagem com o objetivo de fornecer um ponto de referência meteorológico consistente. Ela não modela a variabilidade espacial das fazendas eólicas individuais em toda a Bahia.

O extrator utiliza cache, tentativas de repetição, validação explícita e propaga falhas em vez de retornar silenciosamente conjuntos de dados inválidos.

### Capacidade eólica instalada — ABEEólica INFOVENTO

A capacidade eólica instalada não é mais incorporada diretamente no código de engenharia de features.

A fonte canônica agora é o conjunto de dados de referência versionado:

`data/reference/bahia_wind_capacity_checkpoints.csv`

Cada checkpoint distingue três conceitos temporais:

* `reference_date`: período histórico/data à qual a capacidade relatada se refere;
* `publication_date`: data em que o boletim INFOVENTO foi publicado;
* `available_from`: data mais antiga na qual o pipeline tem permissão para usar o valor.

Para um processamento estrito point-in-time (ponto no tempo):

`available_from = publication_date`

A capacidade é representada como uma função degrau causal usando o último checkpoint disponível em cada carimbo de tempo (timestamp).

Nenhuma interpolação linear ou preenchimento retroativo é realizado. Portanto, um boletim futuro nunca pode ser usado para reconstruir uma observação passada durante o treinamento ou avaliação do modelo.

### Reconciliação com a ANEEL

Os registros de operação comercial da ANEEL são retidos como uma fonte de diagnóstico independente.

Eles são representados como eventos de capacidade no nível da unidade geradora e são usados apenas para reconciliação.

Eles não são usados para reconstruir a série canônica `capacidade_mw` porque o histórico de eventos não reproduz a capacidade instalada total relatada pelo INFOVENTO.

### Alvo do fator de capacidade (Target)

O alvo normalizado permanece:

$$\mathrm{target\_fc} = \frac{\mathrm{wind\_generation\_mw}} {\mathrm{capacidade\_mw}}$$

onde `capacidade_mw` agora vem exclusivamente da série de checkpoints causais do INFOVENTO.

### Feature temporal móvel (rolling)

A feature `wind_speed_roll_mean_3h` também foi revisada.

A janela móvel (rolling window) agora é baseada em três horas cronológicas (`3h`) em vez das três linhas anteriores do dataframe.

Isso evita que a feature abranja grandes lacunas temporais causadas por observações ausentes ou descartadas.

### Validação

A camada de dados revisada é protegida por testes automatizados que cobrem:

* carregamento da referência canônica de capacidade e validação de schema;
* invariantes temporais de checkpoint;
* atribuição de capacidade causal;
* ausência de interpolação e look-ahead (vazamento de dados futuros);
* comportamento móvel cronológico de 3 horas;
* tratamento de horas incompletas do ONS;
* contratos de extração do Open-Meteo;
* extração de eventos da ANEEL e preservação da unidade geradora.

A suíte completa de testes do projeto passou com sucesso após essas mudanças.

### Implicação para o experimento de simplificação do ensemble

As arquiteturas `LGBM + XGB + RF` e `LGBM + RF` devem ser comparadas usando o mesmo conjunto de dados regenerado sob o contrato de dados revisado da v0.2.0.

Os resultados produzidos a partir do snapshot Gold anterior não são misturados com o novo experimento.

A comparação de arquiteturas só começará após os novos snapshots de treinamento e OOT terem sido gerados a partir do pipeline atualizado de extração, proveniência e engenharia de features.


## 6. Snapshot temporal

Preencher **somente depois da expansão da janela temporal e da regeneração dos snapshots sob o contrato de dados v0.2.0**.

Registrar:

- arquivo de treino;
- arquivo OOT;
- início/fim do treino;
- início/fim do OOT;
- número de amostras;
- número de features;
- identificador do snapshot, quando disponível;
- versão/commit do contrato de dados usado na geração;
- última edição INFOVENTO causalmente disponível no período.


In [ ]:
TRAIN_FILE = None
TEST_FILE = None

df_train = None
df_test = None

In [ ]:
import pandas as pd

def summarize_snapshot(df, name):
    if df is None:
        print(f"{name}: ainda não carregado")
        return

    print(f"{name}: {len(df):,} amostras")

    if "date" in df.columns:
        dates = pd.to_datetime(df["date"])
        print(f"  início: {dates.min()}")
        print(f"  fim:    {dates.max()}")

summarize_snapshot(df_train, "TRAIN")
summarize_snapshot(df_test, "OOT")

## 7. Sanity checks antes da execução

Antes de iniciar Optuna:

1. `df_train` e `df_test` devem estar carregados;
2. `TRAIN_FILE` e `TEST_FILE` devem corresponder aos dados usados;
3. os limites temporais de treino e OOT devem ser idênticos entre as arquiteturas;
4. o contrato de features deve permanecer inalterado;
5. o OOT não pode participar da otimização;
6. o orçamento Optuna deve ser idêntico;
7. os snapshots devem ter sido regenerados usando o contrato de provenance de capacidade da v0.2.0;
8. nenhuma observação pode usar um checkpoint de capacidade antes de seu `available_from`;
9. horas ONS incompletas não podem compor o target estadual agregado;
10. as duas arquiteturas devem consumir exatamente os mesmos artefatos regenerados de treino e OOT.


In [ ]:
def validate_experiment_inputs():
    assert df_train is not None, "df_train ainda não foi carregado."
    assert df_test is not None, "df_test ainda não foi carregado."
    assert TRAIN_FILE, "TRAIN_FILE ainda não foi definido."
    assert TEST_FILE, "TEST_FILE ainda não foi definido."

    required_columns = {
        "target_fc",
        "wind_generation_mw",
        "capacidade_mw",
    }

    for name, df in {"train": df_train, "oot": df_test}.items():
        missing = required_columns - set(df.columns)
        assert not missing, (
            f"{name}: colunas obrigatórias ausentes: {sorted(missing)}"
        )

    print("Inputs experimentais validados.")

## 8. Otimização — arquitetura completa

In [ ]:
full_optimization = None

if RUN_EXPERIMENTS:
    validate_experiment_inputs()

    full_optimization = run_optimization(
        df_train=df_train,
        train_file=TRAIN_FILE,
        n_trials=EXPERIMENT_CONFIG["n_trials"],
        n_splits=EXPERIMENT_CONFIG["n_splits"],
        stacking_n_splits=EXPERIMENT_CONFIG["stacking_n_splits"],
        enabled_estimators=ARCHITECTURES["full"],
    )

full_optimization

## 9. Treinamento — arquitetura completa

In [ ]:
full_training = None

if RUN_EXPERIMENTS:
    full_run_id, full_oot_mae_mw = train_stacking_regressor(
        df_train=df_train,
        df_test=df_test,
        train_file=TRAIN_FILE,
        test_file=TEST_FILE,
        best_params=full_optimization["best_params"],
        optimization_run_id=full_optimization["optimization_run_id"],
        enabled_estimators=ARCHITECTURES["full"],
    )

    full_training = {
        "run_id": full_run_id,
        "oot_mae_mw": full_oot_mae_mw,
    }

full_training

## 10. Otimização — arquitetura simplificada

O estudo reduzido deve conter apenas hiperparâmetros `lgb_*` e `rf_*`.
Nenhum `xgb_*` deve aparecer no search space ou no `best_params`.

In [ ]:
simplified_optimization = None

if RUN_EXPERIMENTS:
    validate_experiment_inputs()

    simplified_optimization = run_optimization(
        df_train=df_train,
        train_file=TRAIN_FILE,
        n_trials=EXPERIMENT_CONFIG["n_trials"],
        n_splits=EXPERIMENT_CONFIG["n_splits"],
        stacking_n_splits=EXPERIMENT_CONFIG["stacking_n_splits"],
        enabled_estimators=ARCHITECTURES["simplified"],
    )

simplified_optimization

## 11. Treinamento — arquitetura simplificada

In [ ]:
simplified_training = None

if RUN_EXPERIMENTS:
    simplified_run_id, simplified_oot_mae_mw = train_stacking_regressor(
        df_train=df_train,
        df_test=df_test,
        train_file=TRAIN_FILE,
        test_file=TEST_FILE,
        best_params=simplified_optimization["best_params"],
        optimization_run_id=simplified_optimization["optimization_run_id"],
        enabled_estimators=ARCHITECTURES["simplified"],
    )

    simplified_training = {
        "run_id": simplified_run_id,
        "oot_mae_mw": simplified_oot_mae_mw,
    }

simplified_training

## 12. Resultados a coletar

| Dimensão | Full | Simplified |
|---|---:|---:|
| Base estimators | LGBM + XGB + RF | LGBM + RF |
| Optuna trials | — | — |
| Outer CV splits | — | — |
| Inner CV splits | — | — |
| Best CV nMAE mean (%) | — | — |
| Best CV nMAE std (%) | — | — |
| OOT MAE (MW) | — | — |
| OOT nMAE (%) | — | — |
| OOT MAE FC (%) | — | — |
| OOT R² | — | — |
| Meta-weight LGBM | — | — |
| Meta-weight XGB | — | N/A |
| Meta-weight RF | — | — |
| Optimization time | — | — |
| Training time | — | — |

A decisão final não deve usar uma única métrica isolada.

## 13. Critérios de interpretação

A simplificação ganha evidência quando, sob o **mesmo protocolo**:

- o desempenho OOT permanece comparável;
- a dispersão temporal de CV não piora de forma relevante;
- o modelo reduzido não apresenta sinais de instabilidade;
- o XGBoost continua oferecendo contribuição marginal no ensemble completo;
- a redução de complexidade melhora custo, tempo ou governança operacional.

Se a janela expandida fizer o XGBoost voltar a contribuir de forma relevante, a arquitetura completa deve permanecer candidata.

## 14. Quality Gate e governança

A Quality Gate permanece separada da análise experimental.

Registrar depois:

```text
full:
  quality_gate_result = ...

simplified:
  quality_gate_result = ...
```

Uma eventual rejeição automática não deve ser reescrita como aprovação. Qualquer promoção manual futura deve preservar o resultado original e registrar explicitamente `manual_override`.

## 15. Conclusão experimental

**Preencher somente após a execução com a janela expandida.**

Perguntas:

1. O XGBoost continuou com contribuição aproximadamente nula?
2. O ensemble simplificado preservou nMAE OOT?
3. Houve alteração relevante no CV mean/std?
4. Houve redução mensurável de tempo/custo?
5. A simplificação é sustentada por evidência suficiente para entrar na `v0.2.0`?
6. O Registry/serving precisa de migração de nomenclatura ou governança?

## 16. Próximos passos

Após concluir o experimento:

- atualizar o Model Card com a decisão arquitetural e o contrato de dados v0.2.0;
- resumir a decisão no notebook técnico principal;
- registrar os snapshots Gold regenerados usados no benchmark;
- decidir se a branch experimental entra na `v0.2.0`;
- manter separada a futura fase **Cloud Infrastructure & IaC**.
